# 第107章 物流延期风险预测项目

使用 Olist 巴西电商公开数据，以物流延期分类为主线，学习多表建模、预测时点、数据泄漏、类别不平衡和业务阈值。

## 项目背景

目标是在订单创建后预测是否会晚于预计日期签收。实际发货、实际签收和最终订单状态只能用于构造样本或标签，不能作为预测特征。

## 学习目标

- 理解订单、明细、客户和卖家表的粒度
- 构造订单级延期标签并排除事后字段
- 使用时间顺序划分模拟未来预测
- 比较概率基线、逻辑回归和随机森林
- 使用PR-AUC、Top-K、错误切片与特征重要性评价模型

## 本章教学增强提示

- 模块：**机器学习**；本章结果重点：预测时点、数据泄漏、基线、评估指标、错误样本和模型限制。
- 本章的“课堂自检”用于即时验证理解，不再作为独立章节作业提交。
- 阅读输出时，请同时记录输入口径、关键中间结果和结论边界。

## 本章方法与函数详解

下面按方法逐项学习。每个方法都有独立说明和独立代码单元格，便于单独运行、修改参数并观察结果。

### pd.read_csv

读取 CSV 文件。

- **调用形式**：`import pandas as pd`
- **返回结果**：DataFrame
- **注意事项**：真实文件中还需确认编码和缺失值

先运行下面的最小示例，再回到本章案例观察它在真实数据中的用法。

<!-- 教学增强：方法字段 -->

#### 方法说明

- **作用**：说明 `pd.read_csv` 在本章任务中解决的问题。
- **调用形式**：以当前代码 Cell 为准，补充对象、参数和默认值。
- **参数**：先从下方独立代码 Cell 标出输入参数、默认值、数据类型和是否必填。
- **返回值**：返回处理后的对象、数值、布尔值或结果数组，需用 `type()`、`shape` 或 `print()` 核对。
- **原对象是否改变**：原对象通常不变，返回新对象。
- **错误与边界**：检查空输入、缺失值、类型不匹配、越界、形状不一致或格式不匹配；文件和图形方法还要检查路径与输出副作用。
- **相近方法区别**：与本章相近方法比较输入、返回值、是否修改原对象和适用场景。

#### 学习动作

先独立运行下一个代码 Cell，记录输入、输出、类型和形状；再修改一个参数，比较结果变化，并写出“观察到—说明—限制—下一步”。

In [ ]:
import pandas as pd
from io import StringIO
content = "name,score\nA,80\nB,90"
df = pd.read_csv(StringIO(content))
print(df)

### pd.Series

创建一维带标签数据。

- **调用形式**：`import pandas as pd`
- **返回结果**：Series
- **注意事项**：索引会参与对齐

先运行下面的最小示例，再回到本章案例观察它在真实数据中的用法。

<!-- 教学增强：方法字段 -->

#### 方法说明

- **作用**：说明 `pd.Series` 在本章任务中解决的问题。
- **调用形式**：以当前代码 Cell 为准，补充对象、参数和默认值。
- **参数**：先从下方独立代码 Cell 标出输入参数、默认值、数据类型和是否必填。
- **返回值**：返回处理后的对象、数值、布尔值或结果数组，需用 `type()`、`shape` 或 `print()` 核对。
- **原对象是否改变**：原对象通常不变，返回新对象。
- **错误与边界**：检查空输入、缺失值、类型不匹配、越界、形状不一致或格式不匹配；文件和图形方法还要检查路径与输出副作用。
- **相近方法区别**：与本章相近方法比较输入、返回值、是否修改原对象和适用场景。

#### 学习动作

先独立运行下一个代码 Cell，记录输入、输出、类型和形状；再修改一个参数，比较结果变化，并写出“观察到—说明—限制—下一步”。

In [ ]:
import pandas as pd
s = pd.Series([80, 90], index=["A", "B"])
print(s)

### len

获取对象长度。

- **调用形式**：`text = "Python"`
- **返回结果**：整数
- **注意事项**：字符串长度是字符数量

先运行下面的最小示例，再回到本章案例观察它在真实数据中的用法。

<!-- 教学增强：方法字段 -->

#### 方法说明

- **作用**：说明 `len` 在本章任务中解决的问题。
- **调用形式**：以当前代码 Cell 为准，补充对象、参数和默认值。
- **参数**：先从下方独立代码 Cell 标出输入参数、默认值、数据类型和是否必填。
- **返回值**：返回处理后的对象、数值、布尔值或结果数组，需用 `type()`、`shape` 或 `print()` 核对。
- **原对象是否改变**：原对象通常不变，返回新对象。
- **错误与边界**：检查空输入、缺失值、类型不匹配、越界、形状不一致或格式不匹配；文件和图形方法还要检查路径与输出副作用。
- **相近方法区别**：与本章相近方法比较输入、返回值、是否修改原对象和适用场景。

#### 学习动作

先独立运行下一个代码 Cell，记录输入、输出、类型和形状；再修改一个参数，比较结果变化，并写出“观察到—说明—限制—下一步”。

In [ ]:
text = "Python"
print("字符数:", len(text))
print("列表长度:", len([1, 2, 3]))

### sum

计算可迭代对象总和。

- **调用形式**：`values = [10, 20, 30]`
- **返回结果**：数值
- **注意事项**：空序列可结合 start 参数

先运行下面的最小示例，再回到本章案例观察它在真实数据中的用法。

<!-- 教学增强：方法字段 -->

#### 方法说明

- **作用**：说明 `sum` 在本章任务中解决的问题。
- **调用形式**：以当前代码 Cell 为准，补充对象、参数和默认值。
- **参数**：先从下方独立代码 Cell 标出输入参数、默认值、数据类型和是否必填。
- **返回值**：返回处理后的对象、数值、布尔值或结果数组，需用 `type()`、`shape` 或 `print()` 核对。
- **原对象是否改变**：原对象通常不变，返回新对象。
- **错误与边界**：检查空输入、缺失值、类型不匹配、越界、形状不一致或格式不匹配；文件和图形方法还要检查路径与输出副作用。
- **相近方法区别**：与本章相近方法比较输入、返回值、是否修改原对象和适用场景。

#### 学习动作

先独立运行下一个代码 Cell，记录输入、输出、类型和形状；再修改一个参数，比较结果变化，并写出“观察到—说明—限制—下一步”。

In [ ]:
values = [10, 20, 30]
print(sum(values))

### items.groupby

按键分组。

- **调用形式**：`import pandas as pd`
- **返回结果**：GroupBy 或聚合结果
- **注意事项**：必须配合聚合或转换

先运行下面的最小示例，再回到本章案例观察它在真实数据中的用法。

<!-- 教学增强：方法字段 -->

#### 方法说明

- **作用**：说明 `items.groupby` 在本章任务中解决的问题。
- **调用形式**：以当前代码 Cell 为准，补充对象、参数和默认值。
- **参数**：数据对象、字段/条件、窗口/频率或缺失值策略；必须说明默认值和数据粒度。
- **返回值**：返回处理后的对象、数值、布尔值或结果数组，需用 `type()`、`shape` 或 `print()` 核对。
- **原对象是否改变**：原对象通常不变，返回新对象。
- **错误与边界**：检查空输入、缺失值、类型不匹配、越界、形状不一致或格式不匹配；文件和图形方法还要检查路径与输出副作用。
- **相近方法区别**：与本章相近方法比较输入、返回值、是否修改原对象和适用场景。

#### 学习动作

先独立运行下一个代码 Cell，记录输入、输出、类型和形状；再修改一个参数，比较结果变化，并写出“观察到—说明—限制—下一步”。

In [ ]:
import pandas as pd
df = pd.DataFrame({"region": ["东", "东", "南"], "sales": [10, 20, 15]})
print(df.groupby("region")["sales"].sum())

### agg

对分组结果进行多指标聚合。

- **调用形式**：`import pandas as pd`
- **返回结果**：Series 或 DataFrame
- **注意事项**：给输出列命名

先运行下面的最小示例，再回到本章案例观察它在真实数据中的用法。

<!-- 教学增强：方法字段 -->

#### 方法说明

- **作用**：说明 `agg` 在本章任务中解决的问题。
- **调用形式**：以当前代码 Cell 为准，补充对象、参数和默认值。
- **参数**：数据对象、字段/条件、窗口/频率或缺失值策略；必须说明默认值和数据粒度。
- **返回值**：返回处理后的对象、数值、布尔值或结果数组，需用 `type()`、`shape` 或 `print()` 核对。
- **原对象是否改变**：原对象通常不变，返回新对象。
- **错误与边界**：检查空输入、缺失值、类型不匹配、越界、形状不一致或格式不匹配；文件和图形方法还要检查路径与输出副作用。
- **相近方法区别**：与本章相近方法比较输入、返回值、是否修改原对象和适用场景。

#### 学习动作

先独立运行下一个代码 Cell，记录输入、输出、类型和形状；再修改一个参数，比较结果变化，并写出“观察到—说明—限制—下一步”。

In [ ]:
import pandas as pd
df = pd.DataFrame({"region": ["东", "东", "南"], "sales": [10, 20, 15]})
print(df.groupby("region").agg(total=("sales", "sum"), average=("sales", "mean")))

### head

查看前几行。

- **调用形式**：`import pandas as pd`
- **返回结果**：DataFrame
- **注意事项**：只是抽样查看

先运行下面的最小示例，再回到本章案例观察它在真实数据中的用法。

<!-- 教学增强：方法字段 -->

#### 方法说明

- **作用**：说明 `head` 在本章任务中解决的问题。
- **调用形式**：以当前代码 Cell 为准，补充对象、参数和默认值。
- **参数**：数据对象、字段/条件、窗口/频率或缺失值策略；必须说明默认值和数据粒度。
- **返回值**：返回处理后的对象、数值、布尔值或结果数组，需用 `type()`、`shape` 或 `print()` 核对。
- **原对象是否改变**：原对象通常不变，返回新对象。
- **错误与边界**：检查空输入、缺失值、类型不匹配、越界、形状不一致或格式不匹配；文件和图形方法还要检查路径与输出副作用。
- **相近方法区别**：与本章相近方法比较输入、返回值、是否修改原对象和适用场景。

#### 学习动作

先独立运行下一个代码 Cell，记录输入、输出、类型和形状；再修改一个参数，比较结果变化，并写出“观察到—说明—限制—下一步”。

In [ ]:
import pandas as pd
df = pd.DataFrame({"score": [80, 90, 70]})
print(df.head(2))

### model_df.query

用表达式筛选行。

- **调用形式**：`import pandas as pd`
- **返回结果**：DataFrame
- **注意事项**：表达式中的列名要正确

先运行下面的最小示例，再回到本章案例观察它在真实数据中的用法。

<!-- 教学增强：方法字段 -->

#### 方法说明

- **作用**：说明 `model_df.query` 在本章任务中解决的问题。
- **调用形式**：以当前代码 Cell 为准，补充对象、参数和默认值。
- **参数**：数据对象、字段/条件、窗口/频率或缺失值策略；必须说明默认值和数据粒度。
- **返回值**：返回处理后的对象、数值、布尔值或结果数组，需用 `type()`、`shape` 或 `print()` 核对。
- **原对象是否改变**：原对象通常不变，返回新对象。
- **错误与边界**：检查空输入、缺失值、类型不匹配、越界、形状不一致或格式不匹配；文件和图形方法还要检查路径与输出副作用。
- **相近方法区别**：与本章相近方法比较输入、返回值、是否修改原对象和适用场景。

#### 学习动作

先独立运行下一个代码 Cell，记录输入、输出、类型和形状；再修改一个参数，比较结果变化，并写出“观察到—说明—限制—下一步”。

In [ ]:
import pandas as pd
df = pd.DataFrame({"score": [60, 80, 90]})
print(df.query("score >= 80"))

### astype

转换数据类型。

- **调用形式**：`import pandas as pd`
- **返回结果**：新对象
- **注意事项**：不兼容值会失败

先运行下面的最小示例，再回到本章案例观察它在真实数据中的用法。

<!-- 教学增强：方法字段 -->

#### 方法说明

- **作用**：说明 `astype` 在本章任务中解决的问题。
- **调用形式**：以当前代码 Cell 为准，补充对象、参数和默认值。
- **参数**：输入数组、目标形状、数据类型、条件和步长等参数；注意形状和 dtype。
- **返回值**：返回处理后的对象、数值、布尔值或结果数组，需用 `type()`、`shape` 或 `print()` 核对。
- **原对象是否改变**：原对象通常不变，返回新对象。
- **错误与边界**：检查空输入、缺失值、类型不匹配、越界、形状不一致或格式不匹配；文件和图形方法还要检查路径与输出副作用。
- **相近方法区别**：与本章相近方法比较输入、返回值、是否修改原对象和适用场景。

#### 学习动作

先独立运行下一个代码 Cell，记录输入、输出、类型和形状；再修改一个参数，比较结果变化，并写出“观察到—说明—限制—下一步”。

In [ ]:
import pandas as pd
s = pd.Series(["80", "90"])
print(s.astype(int))

### model_df.sort_values

按列排序。

- **调用形式**：`import pandas as pd`
- **返回结果**：新 DataFrame
- **注意事项**：检查排序方向

先运行下面的最小示例，再回到本章案例观察它在真实数据中的用法。

<!-- 教学增强：方法字段 -->

#### 方法说明

- **作用**：说明 `model_df.sort_values` 在本章任务中解决的问题。
- **调用形式**：以当前代码 Cell 为准，补充对象、参数和默认值。
- **参数**：数据对象、字段/条件、窗口/频率或缺失值策略；必须说明默认值和数据粒度。
- **返回值**：返回处理后的对象、数值、布尔值或结果数组，需用 `type()`、`shape` 或 `print()` 核对。
- **原对象是否改变**：原对象通常不变，返回新对象。
- **错误与边界**：检查空输入、缺失值、类型不匹配、越界、形状不一致或格式不匹配；文件和图形方法还要检查路径与输出副作用。
- **相近方法区别**：与本章相近方法比较输入、返回值、是否修改原对象和适用场景。

#### 学习动作

先独立运行下一个代码 Cell，记录输入、输出、类型和形状；再修改一个参数，比较结果变化，并写出“观察到—说明—限制—下一步”。

In [ ]:
import pandas as pd
df = pd.DataFrame({"score": [80, 90, 70]})
print(df.sort_values("score", ascending=False))

### model_df.groupby

按键分组。

- **调用形式**：`import pandas as pd`
- **返回结果**：GroupBy 或聚合结果
- **注意事项**：必须配合聚合或转换

先运行下面的最小示例，再回到本章案例观察它在真实数据中的用法。

<!-- 教学增强：方法字段 -->

#### 方法说明

- **作用**：说明 `model_df.groupby` 在本章任务中解决的问题。
- **调用形式**：以当前代码 Cell 为准，补充对象、参数和默认值。
- **参数**：数据对象、字段/条件、窗口/频率或缺失值策略；必须说明默认值和数据粒度。
- **返回值**：返回处理后的对象、数值、布尔值或结果数组，需用 `type()`、`shape` 或 `print()` 核对。
- **原对象是否改变**：原对象通常不变，返回新对象。
- **错误与边界**：检查空输入、缺失值、类型不匹配、越界、形状不一致或格式不匹配；文件和图形方法还要检查路径与输出副作用。
- **相近方法区别**：与本章相近方法比较输入、返回值、是否修改原对象和适用场景。

#### 学习动作

先独立运行下一个代码 Cell，记录输入、输出、类型和形状；再修改一个参数，比较结果变化，并写出“观察到—说明—限制—下一步”。

In [ ]:
import pandas as pd
df = pd.DataFrame({"region": ["东", "东", "南"], "sales": [10, 20, 15]})
print(df.groupby("region")["sales"].sum())

### late.agg

对分组结果进行多指标聚合。

- **调用形式**：`import pandas as pd`
- **返回结果**：Series 或 DataFrame
- **注意事项**：给输出列命名

先运行下面的最小示例，再回到本章案例观察它在真实数据中的用法。

<!-- 教学增强：方法字段 -->

#### 方法说明

- **作用**：说明 `late.agg` 在本章任务中解决的问题。
- **调用形式**：以当前代码 Cell 为准，补充对象、参数和默认值。
- **参数**：数据对象、字段/条件、窗口/频率或缺失值策略；必须说明默认值和数据粒度。
- **返回值**：返回处理后的对象、数值、布尔值或结果数组，需用 `type()`、`shape` 或 `print()` 核对。
- **原对象是否改变**：原对象通常不变，返回新对象。
- **错误与边界**：检查空输入、缺失值、类型不匹配、越界、形状不一致或格式不匹配；文件和图形方法还要检查路径与输出副作用。
- **相近方法区别**：与本章相近方法比较输入、返回值、是否修改原对象和适用场景。

#### 学习动作

先独立运行下一个代码 Cell，记录输入、输出、类型和形状；再修改一个参数，比较结果变化，并写出“观察到—说明—限制—下一步”。

In [ ]:
import pandas as pd
df = pd.DataFrame({"region": ["东", "东", "南"], "sales": [10, 20, 15]})
print(df.groupby("region").agg(total=("sales", "sum"), average=("sales", "mean")))

### month_rate.round

按指定小数位舍入。

- **调用形式**：`amount = 1280.567`
- **返回结果**：数值
- **注意事项**：明确是展示舍入还是计算舍入

先运行下面的最小示例，再回到本章案例观察它在真实数据中的用法。

<!-- 教学增强：方法字段 -->

#### 方法说明

- **作用**：说明 `month_rate.round` 在本章任务中解决的问题。
- **调用形式**：以当前代码 Cell 为准，补充对象、参数和默认值。
- **参数**：先从下方独立代码 Cell 标出输入参数、默认值、数据类型和是否必填。
- **返回值**：返回处理后的对象、数值、布尔值或结果数组，需用 `type()`、`shape` 或 `print()` 核对。
- **原对象是否改变**：原对象通常不变，返回新对象。
- **错误与边界**：检查空输入、缺失值、类型不匹配、越界、形状不一致或格式不匹配；文件和图形方法还要检查路径与输出副作用。
- **相近方法区别**：与本章相近方法比较输入、返回值、是否修改原对象和适用场景。

#### 学习动作

先独立运行下一个代码 Cell，记录输入、输出、类型和形状；再修改一个参数，比较结果变化，并写出“观察到—说明—限制—下一步”。

In [ ]:
amount = 1280.567
print(round(amount, 2))

### promise_rate.round

按指定小数位舍入。

- **调用形式**：`amount = 1280.567`
- **返回结果**：数值
- **注意事项**：明确是展示舍入还是计算舍入

先运行下面的最小示例，再回到本章案例观察它在真实数据中的用法。

<!-- 教学增强：方法字段 -->

#### 方法说明

- **作用**：说明 `promise_rate.round` 在本章任务中解决的问题。
- **调用形式**：以当前代码 Cell 为准，补充对象、参数和默认值。
- **参数**：先从下方独立代码 Cell 标出输入参数、默认值、数据类型和是否必填。
- **返回值**：返回处理后的对象、数值、布尔值或结果数组，需用 `type()`、`shape` 或 `print()` 核对。
- **原对象是否改变**：原对象通常不变，返回新对象。
- **错误与边界**：检查空输入、缺失值、类型不匹配、越界、形状不一致或格式不匹配；文件和图形方法还要检查路径与输出副作用。
- **相近方法区别**：与本章相近方法比较输入、返回值、是否修改原对象和适用场景。

#### 学习动作

先独立运行下一个代码 Cell，记录输入、输出、类型和形状；再修改一个参数，比较结果变化，并写出“观察到—说明—限制—下一步”。

In [ ]:
amount = 1280.567
print(round(amount, 2))

### max

取得最大值。

- **调用形式**：`values = [10, 20, 5]`
- **返回结果**：元素中的最大值
- **注意事项**：空序列会失败

先运行下面的最小示例，再回到本章案例观察它在真实数据中的用法。

<!-- 教学增强：方法字段 -->

#### 方法说明

- **作用**：说明 `max` 在本章任务中解决的问题。
- **调用形式**：以当前代码 Cell 为准，补充对象、参数和默认值。
- **参数**：先从下方独立代码 Cell 标出输入参数、默认值、数据类型和是否必填。
- **返回值**：返回处理后的对象、数值、布尔值或结果数组，需用 `type()`、`shape` 或 `print()` 核对。
- **原对象是否改变**：原对象通常不变，返回新对象。
- **错误与边界**：检查空输入、缺失值、类型不匹配、越界、形状不一致或格式不匹配；文件和图形方法还要检查路径与输出副作用。
- **相近方法区别**：与本章相近方法比较输入、返回值、是否修改原对象和适用场景。

#### 学习动作

先独立运行下一个代码 Cell，记录输入、输出、类型和形状；再修改一个参数，比较结果变化，并写出“观察到—说明—限制—下一步”。

In [ ]:
values = [10, 20, 5]
print(max(values))

### int

把值转换为整数。

- **调用形式**：`text = "128"`
- **返回结果**：整数
- **注意事项**：小数文本不能直接用 int 转换

先运行下面的最小示例，再回到本章案例观察它在真实数据中的用法。

<!-- 教学增强：方法字段 -->

#### 方法说明

- **作用**：说明 `int` 在本章任务中解决的问题。
- **调用形式**：以当前代码 Cell 为准，补充对象、参数和默认值。
- **参数**：先从下方独立代码 Cell 标出输入参数、默认值、数据类型和是否必填。
- **返回值**：返回处理后的对象、数值、布尔值或结果数组，需用 `type()`、`shape` 或 `print()` 核对。
- **原对象是否改变**：原对象通常不变，返回新对象。
- **错误与边界**：检查空输入、缺失值、类型不匹配、越界、形状不一致或格式不匹配；文件和图形方法还要检查路径与输出副作用。
- **相近方法区别**：与本章相近方法比较输入、返回值、是否修改原对象和适用场景。

#### 学习动作

先独立运行下一个代码 Cell，记录输入、输出、类型和形状；再修改一个参数，比较结果变化，并写出“观察到—说明—限制—下一步”。

In [ ]:
text = "128"
value = int(text)
print(value, type(value).__name__)

### train.order_purchase_timestamp.max

取得最大值。

- **调用形式**：`values = [10, 20, 5]`
- **返回结果**：元素中的最大值
- **注意事项**：空序列会失败

先运行下面的最小示例，再回到本章案例观察它在真实数据中的用法。

<!-- 教学增强：方法字段 -->

#### 方法说明

- **作用**：说明 `train.order_purchase_timestamp.max` 在本章任务中解决的问题。
- **调用形式**：以当前代码 Cell 为准，补充对象、参数和默认值。
- **参数**：先从下方独立代码 Cell 标出输入参数、默认值、数据类型和是否必填。
- **返回值**：返回处理后的对象、数值、布尔值或结果数组，需用 `type()`、`shape` 或 `print()` 核对。
- **原对象是否改变**：原对象通常不变，返回新对象。
- **错误与边界**：检查空输入、缺失值、类型不匹配、越界、形状不一致或格式不匹配；文件和图形方法还要检查路径与输出副作用。
- **相近方法区别**：与本章相近方法比较输入、返回值、是否修改原对象和适用场景。

#### 学习动作

先独立运行下一个代码 Cell，记录输入、输出、类型和形状；再修改一个参数，比较结果变化，并写出“观察到—说明—限制—下一步”。

In [ ]:
values = [10, 20, 5]
print(max(values))

### test.order_purchase_timestamp.min

取得最小值。

- **调用形式**：`values = [10, 20, 5]`
- **返回结果**：元素中的最小值
- **注意事项**：空序列会失败

先运行下面的最小示例，再回到本章案例观察它在真实数据中的用法。

<!-- 教学增强：方法字段 -->

#### 方法说明

- **作用**：说明 `test.order_purchase_timestamp.min` 在本章任务中解决的问题。
- **调用形式**：以当前代码 Cell 为准，补充对象、参数和默认值。
- **参数**：先从下方独立代码 Cell 标出输入参数、默认值、数据类型和是否必填。
- **返回值**：返回处理后的对象、数值、布尔值或结果数组，需用 `type()`、`shape` 或 `print()` 核对。
- **原对象是否改变**：原对象通常不变，返回新对象。
- **错误与边界**：检查空输入、缺失值、类型不匹配、越界、形状不一致或格式不匹配；文件和图形方法还要检查路径与输出副作用。
- **相近方法区别**：与本章相近方法比较输入、返回值、是否修改原对象和适用场景。

#### 学习动作

先独立运行下一个代码 Cell，记录输入、输出、类型和形状；再修改一个参数，比较结果变化，并写出“观察到—说明—限制—下一步”。

In [ ]:
values = [10, 20, 5]
print(min(values))

### fit

使用训练数据学习模型参数。

- **调用形式**：`from sklearn.linear_model import LinearRegression`
- **返回结果**：模型对象
- **注意事项**：只能用训练数据拟合

先运行下面的最小示例，再回到本章案例观察它在真实数据中的用法。

<!-- 教学增强：方法字段 -->

#### 方法说明

- **作用**：说明 `fit` 在本章任务中解决的问题。
- **调用形式**：以当前代码 Cell 为准，补充对象、参数和默认值。
- **参数**：特征、目标、切分比例、随机种子或模型参数；训练和评估数据不能混用。
- **返回值**：通常返回已经拟合的模型自身。
- **原对象是否改变**：会改变原对象或模型状态；不要把返回值误认为新对象。
- **错误与边界**：检查空输入、缺失值、类型不匹配、越界、形状不一致或格式不匹配；文件和图形方法还要检查路径与输出副作用。
- **相近方法区别**：与本章相近方法比较输入、返回值、是否修改原对象和适用场景。

#### 学习动作

先独立运行下一个代码 Cell，记录输入、输出、类型和形状；再修改一个参数，比较结果变化，并写出“观察到—说明—限制—下一步”。

In [ ]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit([[1], [2], [3]], [2, 4, 6])
print("模型已拟合")

### round

按指定小数位舍入。

- **调用形式**：`amount = 1280.567`
- **返回结果**：数值
- **注意事项**：明确是展示舍入还是计算舍入

先运行下面的最小示例，再回到本章案例观察它在真实数据中的用法。

<!-- 教学增强：方法字段 -->

#### 方法说明

- **作用**：说明 `round` 在本章任务中解决的问题。
- **调用形式**：以当前代码 Cell 为准，补充对象、参数和默认值。
- **参数**：先从下方独立代码 Cell 标出输入参数、默认值、数据类型和是否必填。
- **返回值**：返回处理后的对象、数值、布尔值或结果数组，需用 `type()`、`shape` 或 `print()` 核对。
- **原对象是否改变**：原对象通常不变，返回新对象。
- **错误与边界**：检查空输入、缺失值、类型不匹配、越界、形状不一致或格式不匹配；文件和图形方法还要检查路径与输出副作用。
- **相近方法区别**：与本章相近方法比较输入、返回值、是否修改原对象和适用场景。

#### 学习动作

先独立运行下一个代码 Cell，记录输入、输出、类型和形状；再修改一个参数，比较结果变化，并写出“观察到—说明—限制—下一步”。

In [ ]:
amount = 1280.567
print(round(amount, 2))

### models.items

遍历字典的键和值。

- **调用形式**：`data = {"name": "Alice", "age": 20}`
- **返回结果**：键值对视图
- **注意事项**：遍历时不要随意改变字典结构

先运行下面的最小示例，再回到本章案例观察它在真实数据中的用法。

<!-- 教学增强：方法字段 -->

#### 方法说明

- **作用**：说明 `models.items` 在本章任务中解决的问题。
- **调用形式**：以当前代码 Cell 为准，补充对象、参数和默认值。
- **参数**：先从下方独立代码 Cell 标出输入参数、默认值、数据类型和是否必填。
- **返回值**：返回处理后的对象、数值、布尔值或结果数组，需用 `type()`、`shape` 或 `print()` 核对。
- **原对象是否改变**：原对象通常不变，返回新对象。
- **错误与边界**：检查空输入、缺失值、类型不匹配、越界、形状不一致或格式不匹配；文件和图形方法还要检查路径与输出副作用。
- **相近方法区别**：与本章相近方法比较输入、返回值、是否修改原对象和适用场景。

#### 学习动作

先独立运行下一个代码 Cell，记录输入、输出、类型和形状；再修改一个参数，比较结果变化，并写出“观察到—说明—限制—下一步”。

In [ ]:
data = {"name": "Alice", "age": 20}
for key, value in data.items():
    print(key, value)

### model.fit

使用训练数据学习模型参数。

- **调用形式**：`from sklearn.linear_model import LinearRegression`
- **返回结果**：模型对象
- **注意事项**：只能用训练数据拟合

先运行下面的最小示例，再回到本章案例观察它在真实数据中的用法。

<!-- 教学增强：方法字段 -->

#### 方法说明

- **作用**：说明 `model.fit` 在本章任务中解决的问题。
- **调用形式**：以当前代码 Cell 为准，补充对象、参数和默认值。
- **参数**：特征、目标、切分比例、随机种子或模型参数；训练和评估数据不能混用。
- **返回值**：通常返回已经拟合的模型自身。
- **原对象是否改变**：会改变原对象或模型状态；不要把返回值误认为新对象。
- **错误与边界**：检查空输入、缺失值、类型不匹配、越界、形状不一致或格式不匹配；文件和图形方法还要检查路径与输出副作用。
- **相近方法区别**：与本章相近方法比较输入、返回值、是否修改原对象和适用场景。

#### 学习动作

先独立运行下一个代码 Cell，记录输入、输出、类型和形状；再修改一个参数，比较结果变化，并写出“观察到—说明—限制—下一步”。

In [ ]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()
model.fit([[1], [2], [3]], [2, 4, 6])
print("模型已拟合")

### rows.append

列表末尾追加一个元素。

- **调用形式**：`values = [10, 20]`
- **返回结果**：返回 None；直接修改原列表
- **注意事项**：一次追加一个元素，不能把多个元素自动展开

先运行下面的最小示例，再回到本章案例观察它在真实数据中的用法。

<!-- 教学增强：方法字段 -->

#### 方法说明

- **作用**：说明 `rows.append` 在本章任务中解决的问题。
- **调用形式**：以当前代码 Cell 为准，补充对象、参数和默认值。
- **参数**：先从下方独立代码 Cell 标出输入参数、默认值、数据类型和是否必填。
- **返回值**：通常返回 None（`pop()` 返回被删除的元素）。
- **原对象是否改变**：会改变原对象或模型状态；不要把返回值误认为新对象。
- **错误与边界**：检查空输入、缺失值、类型不匹配、越界、形状不一致或格式不匹配；文件和图形方法还要检查路径与输出副作用。
- **相近方法区别**：与本章相近方法比较输入、返回值、是否修改原对象和适用场景。

#### 学习动作

先独立运行下一个代码 Cell，记录输入、输出、类型和形状；再修改一个参数，比较结果变化，并写出“观察到—说明—限制—下一步”。

In [ ]:
values = [10, 20]
values.append(30)
print("append 后:", values)

### pd.DataFrame

创建二维表格。

- **调用形式**：`import pandas as pd`
- **返回结果**：DataFrame
- **注意事项**：检查列名和数据类型

先运行下面的最小示例，再回到本章案例观察它在真实数据中的用法。

<!-- 教学增强：方法字段 -->

#### 方法说明

- **作用**：说明 `pd.DataFrame` 在本章任务中解决的问题。
- **调用形式**：以当前代码 Cell 为准，补充对象、参数和默认值。
- **参数**：先从下方独立代码 Cell 标出输入参数、默认值、数据类型和是否必填。
- **返回值**：返回处理后的对象、数值、布尔值或结果数组，需用 `type()`、`shape` 或 `print()` 核对。
- **原对象是否改变**：原对象通常不变，返回新对象。
- **错误与边界**：检查空输入、缺失值、类型不匹配、越界、形状不一致或格式不匹配；文件和图形方法还要检查路径与输出副作用。
- **相近方法区别**：与本章相近方法比较输入、返回值、是否修改原对象和适用场景。

#### 学习动作

先独立运行下一个代码 Cell，记录输入、输出、类型和形状；再修改一个参数，比较结果变化，并写出“观察到—说明—限制—下一步”。

In [ ]:
import pandas as pd
df = pd.DataFrame({"name": ["A", "B"], "score": [80, 90]})
print(df)

### sort_values

按列排序。

- **调用形式**：`import pandas as pd`
- **返回结果**：新 DataFrame
- **注意事项**：检查排序方向

先运行下面的最小示例，再回到本章案例观察它在真实数据中的用法。

<!-- 教学增强：方法字段 -->

#### 方法说明

- **作用**：说明 `sort_values` 在本章任务中解决的问题。
- **调用形式**：以当前代码 Cell 为准，补充对象、参数和默认值。
- **参数**：数据对象、字段/条件、窗口/频率或缺失值策略；必须说明默认值和数据粒度。
- **返回值**：返回处理后的对象、数值、布尔值或结果数组，需用 `type()`、`shape` 或 `print()` 核对。
- **原对象是否改变**：原对象通常不变，返回新对象。
- **错误与边界**：检查空输入、缺失值、类型不匹配、越界、形状不一致或格式不匹配；文件和图形方法还要检查路径与输出副作用。
- **相近方法区别**：与本章相近方法比较输入、返回值、是否修改原对象和适用场景。

#### 学习动作

先独立运行下一个代码 Cell，记录输入、输出、类型和形状；再修改一个参数，比较结果变化，并写出“观察到—说明—限制—下一步”。

In [ ]:
import pandas as pd
df = pd.DataFrame({"score": [80, 90, 70]})
print(df.sort_values("score", ascending=False))

### validation.round

按指定小数位舍入。

- **调用形式**：`amount = 1280.567`
- **返回结果**：数值
- **注意事项**：明确是展示舍入还是计算舍入

先运行下面的最小示例，再回到本章案例观察它在真实数据中的用法。

<!-- 教学增强：方法字段 -->

#### 方法说明

- **作用**：说明 `validation.round` 在本章任务中解决的问题。
- **调用形式**：以当前代码 Cell 为准，补充对象、参数和默认值。
- **参数**：先从下方独立代码 Cell 标出输入参数、默认值、数据类型和是否必填。
- **返回值**：返回处理后的对象、数值、布尔值或结果数组，需用 `type()`、`shape` 或 `print()` 核对。
- **原对象是否改变**：原对象通常不变，返回新对象。
- **错误与边界**：检查空输入、缺失值、类型不匹配、越界、形状不一致或格式不匹配；文件和图形方法还要检查路径与输出副作用。
- **相近方法区别**：与本章相近方法比较输入、返回值、是否修改原对象和适用场景。

#### 学习动作

先独立运行下一个代码 Cell，记录输入、输出、类型和形状；再修改一个参数，比较结果变化，并写出“观察到—说明—限制—下一步”。

In [ ]:
amount = 1280.567
print(round(amount, 2))

### roc_auc_score

计算 ROC-AUC。

- **调用形式**：`from sklearn.metrics import roc_auc_score`
- **返回结果**：浮点数
- **注意事项**：需要连续得分或概率

先运行下面的最小示例，再回到本章案例观察它在真实数据中的用法。

<!-- 教学增强：方法字段 -->

#### 方法说明

- **作用**：说明 `roc_auc_score` 在本章任务中解决的问题。
- **调用形式**：以当前代码 Cell 为准，补充对象、参数和默认值。
- **参数**：先从下方独立代码 Cell 标出输入参数、默认值、数据类型和是否必填。
- **返回值**：返回处理后的对象、数值、布尔值或结果数组，需用 `type()`、`shape` 或 `print()` 核对。
- **原对象是否改变**：原对象通常不变，返回新对象。
- **错误与边界**：检查空输入、缺失值、类型不匹配、越界、形状不一致或格式不匹配；文件和图形方法还要检查路径与输出副作用。
- **相近方法区别**：与本章相近方法比较输入、返回值、是否修改原对象和适用场景。

#### 学习动作

先独立运行下一个代码 Cell，记录输入、输出、类型和形状；再修改一个参数，比较结果变化，并写出“观察到—说明—限制—下一步”。

In [ ]:
from sklearn.metrics import roc_auc_score
print(roc_auc_score([0, 1, 1], [0.1, 0.8, 0.4]))

### ranked.head

查看前几行。

- **调用形式**：`import pandas as pd`
- **返回结果**：DataFrame
- **注意事项**：只是抽样查看

先运行下面的最小示例，再回到本章案例观察它在真实数据中的用法。

<!-- 教学增强：方法字段 -->

#### 方法说明

- **作用**：说明 `ranked.head` 在本章任务中解决的问题。
- **调用形式**：以当前代码 Cell 为准，补充对象、参数和默认值。
- **参数**：数据对象、字段/条件、窗口/频率或缺失值策略；必须说明默认值和数据粒度。
- **返回值**：返回处理后的对象、数值、布尔值或结果数组，需用 `type()`、`shape` 或 `print()` 核对。
- **原对象是否改变**：原对象通常不变，返回新对象。
- **错误与边界**：检查空输入、缺失值、类型不匹配、越界、形状不一致或格式不匹配；文件和图形方法还要检查路径与输出副作用。
- **相近方法区别**：与本章相近方法比较输入、返回值、是否修改原对象和适用场景。

#### 学习动作

先独立运行下一个代码 Cell，记录输入、输出、类型和形状；再修改一个参数，比较结果变化，并写出“观察到—说明—限制—下一步”。

In [ ]:
import pandas as pd
df = pd.DataFrame({"score": [80, 90, 70]})
print(df.head(2))

### threshold_rows.append

列表末尾追加一个元素。

- **调用形式**：`values = [10, 20]`
- **返回结果**：返回 None；直接修改原列表
- **注意事项**：一次追加一个元素，不能把多个元素自动展开

先运行下面的最小示例，再回到本章案例观察它在真实数据中的用法。

<!-- 教学增强：方法字段 -->

#### 方法说明

- **作用**：说明 `threshold_rows.append` 在本章任务中解决的问题。
- **调用形式**：以当前代码 Cell 为准，补充对象、参数和默认值。
- **参数**：先从下方独立代码 Cell 标出输入参数、默认值、数据类型和是否必填。
- **返回值**：通常返回 None（`pop()` 返回被删除的元素）。
- **原对象是否改变**：会改变原对象或模型状态；不要把返回值误认为新对象。
- **错误与边界**：检查空输入、缺失值、类型不匹配、越界、形状不一致或格式不匹配；文件和图形方法还要检查路径与输出副作用。
- **相近方法区别**：与本章相近方法比较输入、返回值、是否修改原对象和适用场景。

#### 学习动作

先独立运行下一个代码 Cell，记录输入、输出、类型和形状；再修改一个参数，比较结果变化，并写出“观察到—说明—限制—下一步”。

In [ ]:
values = [10, 20]
values.append(30)
print("append 后:", values)

### top.probability.min

取得最小值。

- **调用形式**：`values = [10, 20, 5]`
- **返回结果**：元素中的最小值
- **注意事项**：空序列会失败

先运行下面的最小示例，再回到本章案例观察它在真实数据中的用法。

<!-- 教学增强：方法字段 -->

#### 方法说明

- **作用**：说明 `top.probability.min` 在本章任务中解决的问题。
- **调用形式**：以当前代码 Cell 为准，补充对象、参数和默认值。
- **参数**：先从下方独立代码 Cell 标出输入参数、默认值、数据类型和是否必填。
- **返回值**：返回处理后的对象、数值、布尔值或结果数组，需用 `type()`、`shape` 或 `print()` 核对。
- **原对象是否改变**：原对象通常不变，返回新对象。
- **错误与边界**：检查空输入、缺失值、类型不匹配、越界、形状不一致或格式不匹配；文件和图形方法还要检查路径与输出副作用。
- **相近方法区别**：与本章相近方法比较输入、返回值、是否修改原对象和适用场景。

#### 学习动作

先独立运行下一个代码 Cell，记录输入、输出、类型和形状；再修改一个参数，比较结果变化，并写出“观察到—说明—限制—下一步”。

In [ ]:
values = [10, 20, 5]
print(min(values))

### top.actual.sum

计算可迭代对象总和。

- **调用形式**：`values = [10, 20, 30]`
- **返回结果**：数值
- **注意事项**：空序列可结合 start 参数

先运行下面的最小示例，再回到本章案例观察它在真实数据中的用法。

<!-- 教学增强：方法字段 -->

#### 方法说明

- **作用**：说明 `top.actual.sum` 在本章任务中解决的问题。
- **调用形式**：以当前代码 Cell 为准，补充对象、参数和默认值。
- **参数**：先从下方独立代码 Cell 标出输入参数、默认值、数据类型和是否必填。
- **返回值**：返回处理后的对象、数值、布尔值或结果数组，需用 `type()`、`shape` 或 `print()` 核对。
- **原对象是否改变**：原对象通常不变，返回新对象。
- **错误与边界**：检查空输入、缺失值、类型不匹配、越界、形状不一致或格式不匹配；文件和图形方法还要检查路径与输出副作用。
- **相近方法区别**：与本章相近方法比较输入、返回值、是否修改原对象和适用场景。

#### 学习动作

先独立运行下一个代码 Cell，记录输入、输出、类型和形状；再修改一个参数，比较结果变化，并写出“观察到—说明—限制—下一步”。

In [ ]:
values = [10, 20, 30]
print(sum(values))

### ranked.actual.sum

计算可迭代对象总和。

- **调用形式**：`values = [10, 20, 30]`
- **返回结果**：数值
- **注意事项**：空序列可结合 start 参数

先运行下面的最小示例，再回到本章案例观察它在真实数据中的用法。

<!-- 教学增强：方法字段 -->

#### 方法说明

- **作用**：说明 `ranked.actual.sum` 在本章任务中解决的问题。
- **调用形式**：以当前代码 Cell 为准，补充对象、参数和默认值。
- **参数**：先从下方独立代码 Cell 标出输入参数、默认值、数据类型和是否必填。
- **返回值**：返回处理后的对象、数值、布尔值或结果数组，需用 `type()`、`shape` 或 `print()` 核对。
- **原对象是否改变**：原对象通常不变，返回新对象。
- **错误与边界**：检查空输入、缺失值、类型不匹配、越界、形状不一致或格式不匹配；文件和图形方法还要检查路径与输出副作用。
- **相近方法区别**：与本章相近方法比较输入、返回值、是否修改原对象和适用场景。

#### 学习动作

先独立运行下一个代码 Cell，记录输入、输出、类型和形状；再修改一个参数，比较结果变化，并写出“观察到—说明—限制—下一步”。

In [ ]:
values = [10, 20, 30]
print(sum(values))

### metrics.round

按指定小数位舍入。

- **调用形式**：`amount = 1280.567`
- **返回结果**：数值
- **注意事项**：明确是展示舍入还是计算舍入

先运行下面的最小示例，再回到本章案例观察它在真实数据中的用法。

<!-- 教学增强：方法字段 -->

#### 方法说明

- **作用**：说明 `metrics.round` 在本章任务中解决的问题。
- **调用形式**：以当前代码 Cell 为准，补充对象、参数和默认值。
- **参数**：先从下方独立代码 Cell 标出输入参数、默认值、数据类型和是否必填。
- **返回值**：返回处理后的对象、数值、布尔值或结果数组，需用 `type()`、`shape` 或 `print()` 核对。
- **原对象是否改变**：原对象通常不变，返回新对象。
- **错误与边界**：检查空输入、缺失值、类型不匹配、越界、形状不一致或格式不匹配；文件和图形方法还要检查路径与输出副作用。
- **相近方法区别**：与本章相近方法比较输入、返回值、是否修改原对象和适用场景。

#### 学习动作

先独立运行下一个代码 Cell，记录输入、输出、类型和形状；再修改一个参数，比较结果变化，并写出“观察到—说明—限制—下一步”。

In [ ]:
amount = 1280.567
print(round(amount, 2))

### threshold_table.round

按指定小数位舍入。

- **调用形式**：`amount = 1280.567`
- **返回结果**：数值
- **注意事项**：明确是展示舍入还是计算舍入

先运行下面的最小示例，再回到本章案例观察它在真实数据中的用法。

<!-- 教学增强：方法字段 -->

#### 方法说明

- **作用**：说明 `threshold_table.round` 在本章任务中解决的问题。
- **调用形式**：以当前代码 Cell 为准，补充对象、参数和默认值。
- **参数**：先从下方独立代码 Cell 标出输入参数、默认值、数据类型和是否必填。
- **返回值**：返回处理后的对象、数值、布尔值或结果数组，需用 `type()`、`shape` 或 `print()` 核对。
- **原对象是否改变**：原对象通常不变，返回新对象。
- **错误与边界**：检查空输入、缺失值、类型不匹配、越界、形状不一致或格式不匹配；文件和图形方法还要检查路径与输出副作用。
- **相近方法区别**：与本章相近方法比较输入、返回值、是否修改原对象和适用场景。

#### 学习动作

先独立运行下一个代码 Cell，记录输入、输出、类型和形状；再修改一个参数，比较结果变化，并写出“观察到—说明—限制—下一步”。

In [ ]:
amount = 1280.567
print(round(amount, 2))

### error_df.groupby

按键分组。

- **调用形式**：`import pandas as pd`
- **返回结果**：GroupBy 或聚合结果
- **注意事项**：必须配合聚合或转换

先运行下面的最小示例，再回到本章案例观察它在真实数据中的用法。

<!-- 教学增强：方法字段 -->

#### 方法说明

- **作用**：说明 `error_df.groupby` 在本章任务中解决的问题。
- **调用形式**：以当前代码 Cell 为准，补充对象、参数和默认值。
- **参数**：数据对象、字段/条件、窗口/频率或缺失值策略；必须说明默认值和数据粒度。
- **返回值**：返回处理后的对象、数值、布尔值或结果数组，需用 `type()`、`shape` 或 `print()` 核对。
- **原对象是否改变**：原对象通常不变，返回新对象。
- **错误与边界**：检查空输入、缺失值、类型不匹配、越界、形状不一致或格式不匹配；文件和图形方法还要检查路径与输出副作用。
- **相近方法区别**：与本章相近方法比较输入、返回值、是否修改原对象和适用场景。

#### 学习动作

先独立运行下一个代码 Cell，记录输入、输出、类型和形状；再修改一个参数，比较结果变化，并写出“观察到—说明—限制—下一步”。

In [ ]:
import pandas as pd
df = pd.DataFrame({"region": ["东", "东", "南"], "sales": [10, 20, 15]})
print(df.groupby("region")["sales"].sum())

### query

用表达式筛选行。

- **调用形式**：`import pandas as pd`
- **返回结果**：DataFrame
- **注意事项**：表达式中的列名要正确

先运行下面的最小示例，再回到本章案例观察它在真实数据中的用法。

<!-- 教学增强：方法字段 -->

#### 方法说明

- **作用**：说明 `query` 在本章任务中解决的问题。
- **调用形式**：以当前代码 Cell 为准，补充对象、参数和默认值。
- **参数**：数据对象、字段/条件、窗口/频率或缺失值策略；必须说明默认值和数据粒度。
- **返回值**：返回处理后的对象、数值、布尔值或结果数组，需用 `type()`、`shape` 或 `print()` 核对。
- **原对象是否改变**：原对象通常不变，返回新对象。
- **错误与边界**：检查空输入、缺失值、类型不匹配、越界、形状不一致或格式不匹配；文件和图形方法还要检查路径与输出副作用。
- **相近方法区别**：与本章相近方法比较输入、返回值、是否修改原对象和适用场景。

#### 学习动作

先独立运行下一个代码 Cell，记录输入、输出、类型和形状；再修改一个参数，比较结果变化，并写出“观察到—说明—限制—下一步”。

In [ ]:
import pandas as pd
df = pd.DataFrame({"score": [60, 80, 90]})
print(df.query("score >= 80"))

### state_report.head

查看前几行。

- **调用形式**：`import pandas as pd`
- **返回结果**：DataFrame
- **注意事项**：只是抽样查看

先运行下面的最小示例，再回到本章案例观察它在真实数据中的用法。

<!-- 教学增强：方法字段 -->

#### 方法说明

- **作用**：说明 `state_report.head` 在本章任务中解决的问题。
- **调用形式**：以当前代码 Cell 为准，补充对象、参数和默认值。
- **参数**：数据对象、字段/条件、窗口/频率或缺失值策略；必须说明默认值和数据粒度。
- **返回值**：返回处理后的对象、数值、布尔值或结果数组，需用 `type()`、`shape` 或 `print()` 核对。
- **原对象是否改变**：原对象通常不变，返回新对象。
- **错误与边界**：检查空输入、缺失值、类型不匹配、越界、形状不一致或格式不匹配；文件和图形方法还要检查路径与输出副作用。
- **相近方法区别**：与本章相近方法比较输入、返回值、是否修改原对象和适用场景。

#### 学习动作

先独立运行下一个代码 Cell，记录输入、输出、类型和形状；再修改一个参数，比较结果变化，并写出“观察到—说明—限制—下一步”。

In [ ]:
import pandas as pd
df = pd.DataFrame({"score": [80, 90, 70]})
print(df.head(2))

### min

取得最小值。

- **调用形式**：`values = [10, 20, 5]`
- **返回结果**：元素中的最小值
- **注意事项**：空序列会失败

先运行下面的最小示例，再回到本章案例观察它在真实数据中的用法。

<!-- 教学增强：方法字段 -->

#### 方法说明

- **作用**：说明 `min` 在本章任务中解决的问题。
- **调用形式**：以当前代码 Cell 为准，补充对象、参数和默认值。
- **参数**：先从下方独立代码 Cell 标出输入参数、默认值、数据类型和是否必填。
- **返回值**：返回处理后的对象、数值、布尔值或结果数组，需用 `type()`、`shape` 或 `print()` 核对。
- **原对象是否改变**：原对象通常不变，返回新对象。
- **错误与边界**：检查空输入、缺失值、类型不匹配、越界、形状不一致或格式不匹配；文件和图形方法还要检查路径与输出副作用。
- **相近方法区别**：与本章相近方法比较输入、返回值、是否修改原对象和适用场景。

#### 学习动作

先独立运行下一个代码 Cell，记录输入、输出、类型和形状；再修改一个参数，比较结果变化，并写出“观察到—说明—限制—下一步”。

In [ ]:
values = [10, 20, 5]
print(min(values))

### np.linspace

生成指定数量等间隔点。

- **调用形式**：`import numpy as np`
- **返回结果**：ndarray
- **注意事项**：默认包含终点

先运行下面的最小示例，再回到本章案例观察它在真实数据中的用法。

<!-- 教学增强：方法字段 -->

#### 方法说明

- **作用**：说明 `np.linspace` 在本章任务中解决的问题。
- **调用形式**：以当前代码 Cell 为准，补充对象、参数和默认值。
- **参数**：输入数组、目标形状、数据类型、条件和步长等参数；注意形状和 dtype。
- **返回值**：返回处理后的对象、数值、布尔值或结果数组，需用 `type()`、`shape` 或 `print()` 核对。
- **原对象是否改变**：原对象通常不变，返回新对象。
- **错误与边界**：检查空输入、缺失值、类型不匹配、越界、形状不一致或格式不匹配；文件和图形方法还要检查路径与输出副作用。
- **相近方法区别**：与本章相近方法比较输入、返回值、是否修改原对象和适用场景。

#### 学习动作

先独立运行下一个代码 Cell，记录输入、输出、类型和形状；再修改一个参数，比较结果变化，并写出“观察到—说明—限制—下一步”。

In [ ]:
import numpy as np
print(np.linspace(0, 1, 5))

### importance.round

按指定小数位舍入。

- **调用形式**：`amount = 1280.567`
- **返回结果**：数值
- **注意事项**：明确是展示舍入还是计算舍入

先运行下面的最小示例，再回到本章案例观察它在真实数据中的用法。

<!-- 教学增强：方法字段 -->

#### 方法说明

- **作用**：说明 `importance.round` 在本章任务中解决的问题。
- **调用形式**：以当前代码 Cell 为准，补充对象、参数和默认值。
- **参数**：先从下方独立代码 Cell 标出输入参数、默认值、数据类型和是否必填。
- **返回值**：返回处理后的对象、数值、布尔值或结果数组，需用 `type()`、`shape` 或 `print()` 核对。
- **原对象是否改变**：原对象通常不变，返回新对象。
- **错误与边界**：检查空输入、缺失值、类型不匹配、越界、形状不一致或格式不匹配；文件和图形方法还要检查路径与输出副作用。
- **相近方法区别**：与本章相近方法比较输入、返回值、是否修改原对象和适用场景。

#### 学习动作

先独立运行下一个代码 Cell，记录输入、输出、类型和形状；再修改一个参数，比较结果变化，并写出“观察到—说明—限制—下一步”。

In [ ]:
amount = 1280.567
print(round(amount, 2))

### 方法学习检查

- [ ] 能说明每个方法的输入、参数和返回结果
- [ ] 能独立运行对应代码单元格
- [ ] 能修改至少一个参数并解释输出变化
- [ ] 能说明该方法是否修改原对象以及适用边界

<!-- 教学增强：方法字段 -->

#### 方法说明

- **作用**：说明 `方法学习检查` 在本章任务中解决的问题。
- **调用形式**：以当前代码 Cell 为准，补充对象、参数和默认值。
- **参数**：先从下方独立代码 Cell 标出输入参数、默认值、数据类型和是否必填。
- **返回值**：返回处理后的对象、数值、布尔值或结果数组，需用 `type()`、`shape` 或 `print()` 核对。
- **原对象是否改变**：原对象通常不变，返回新对象。
- **错误与边界**：检查空输入、缺失值、类型不匹配、越界、形状不一致或格式不匹配；文件和图形方法还要检查路径与输出副作用。
- **相近方法区别**：与本章相近方法比较输入、返回值、是否修改原对象和适用场景。

#### 学习动作

先独立运行下一个代码 Cell，记录输入、输出、类型和形状；再修改一个参数，比较结果变化，并写出“观察到—说明—限制—下一步”。

## 数据字典

| 字段 | 含义 | 使用说明 |
| --- | --- | --- |
| order_id | 订单主键 | 最终保持一单一行 |
| customer/seller_state | 客户州/卖家州 | 下单时可用类别特征 |
| promise_days | 承诺时长 | 预计送达日减下单日 |
| goods/freight_value | 商品额/运费 | 订单级数值特征 |
| late | 是否延期 | 实际签收晚于预计日 |

## 数据质量检查清单

- 订单主键及明细一对多关系
- 聚合连接后订单唯一
- 日期缺失与承诺天数异常
- 已签收样本的选择口径
- 实际发货和签收字段泄漏
- 延期率随时间和地区变化

## 1. 加载四表并审计数据粒度

先确认每张表的一行代表什么，再决定聚合和连接方式。

In [ ]:
import numpy as np
import pandas as pd

orders = pd.read_csv(
    "/datasets/olist_orders_dataset.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
        "order_delivered_carrier_date",
    ],
)
items = pd.read_csv("/datasets/olist_order_items_dataset.csv")
customers = pd.read_csv("/datasets/olist_customers_dataset.csv")
sellers = pd.read_csv("/datasets/olist_sellers_dataset.csv")
audit = pd.Series(
    {
        "订单行": len(orders),
        "订单键重复": orders.order_id.duplicated().sum(),
        "明细行": len(items),
        "明细订单数": items.order_id.nunique(),
        "客户键重复": customers.customer_id.duplicated().sum(),
        "卖家键重复": sellers.seller_id.duplicated().sum(),
    }
)
print(audit.to_string())

## 2. 聚合明细并连接订单级样本

先把一对多商品明细聚合到订单，再用validate检查连接基数。

In [ ]:
item_agg = items.groupby("order_id").agg(
    item_count=("order_item_id", "size"),
    goods_value=("price", "sum"),
    freight_value=("freight_value", "sum"),
    primary_seller=("seller_id", "first"),
    seller_count=("seller_id", "nunique"),
)
order_level = (
    orders.merge(item_agg, on="order_id", validate="one_to_one")
    .merge(
        customers[["customer_id", "customer_state"]],
        on="customer_id",
        validate="many_to_one",
    )
    .merge(
        sellers[["seller_id", "seller_state"]].rename(
            columns={"seller_id": "primary_seller"}
        ),
        on="primary_seller",
        validate="many_to_one",
    )
)
print("订单编号是否唯一:", order_level.order_id.is_unique)
print(
    "连接后订单:",
    len(order_level),
    "缺少实际签收:",
    order_level.order_delivered_customer_date.isna().sum(),
)
display(
    order_level[
        [
            "order_id",
            "item_count",
            "goods_value",
            "freight_value",
            "seller_count",
        ]
    ].head()
)

## 3. 清洗样本并定义延期标签

标签来自签收结果；建模样本限定为有完整日期的已签收订单，并记录保留率。

In [ ]:
complete = (
    order_level.order_status.eq("delivered")
    & order_level.order_delivered_customer_date.notna()
    & order_level.order_estimated_delivery_date.notna()
)
model_df = order_level.loc[complete].copy()
model_df["promise_days"] = (
    model_df.order_estimated_delivery_date - model_df.order_purchase_timestamp
).dt.total_seconds() / 86400
model_df = model_df.query("promise_days>0").copy()
model_df["late"] = (
    model_df.order_delivered_customer_date
    > model_df.order_estimated_delivery_date
).astype(int)
model_df["month"] = model_df.order_purchase_timestamp.dt.month
model_df["weekday"] = model_df.order_purchase_timestamp.dt.dayofweek
model_df = model_df.sort_values("order_purchase_timestamp").reset_index(
    drop=True
)
print(
    "建模订单:",
    len(model_df),
    "保留率:",
    f"{len(model_df)/len(orders):.1%}",
    "延期率:",
    f"{model_df.late.mean():.2%}",
)

## 4. 探索延期率与类别不平衡

观察月份、承诺时长和订单规模的差异，但不把描述性相关解释为延期原因。

In [ ]:
model_df["promise_group"] = pd.qcut(
    model_df.promise_days, 4, duplicates="drop"
)
month_rate = model_df.groupby("month").late.agg(["size", "mean"])
promise_rate = model_df.groupby("promise_group", observed=True).late.agg(
    ["size", "mean"]
)
print("月份延期率:\n", month_rate.round(3))
print("承诺时长分组延期率:\n", promise_rate.round(3))
print("多数类准确率:", f"{max(model_df.late.mean(),1-model_df.late.mean()):.2%}")

## 5. 泄漏审计与时间顺序划分

用早期订单训练、中期订单验证、晚期订单测试，模拟模型面对未来数据。

In [ ]:
num = [
    "item_count",
    "goods_value",
    "freight_value",
    "seller_count",
    "month",
    "weekday",
    "promise_days",
]
cat = ["customer_state", "seller_state"]
features = num + cat
forbidden = [
    "order_delivered_customer_date",
    "order_delivered_carrier_date",
    "order_status",
]
print("是否使用禁止字段:", bool(set(features) & set(forbidden)))
train_end = int(len(model_df) * 0.64)
val_end = int(len(model_df) * 0.80)
train = model_df.iloc[:train_end]
val = model_df.iloc[train_end:val_end]
test = model_df.iloc[val_end:]
print("禁止字段:", forbidden)
print("训练/验证/测试:", len(train), len(val), len(test))
print("延期率:", *[f"{part.late.mean():.2%}" for part in [train, val, test]])
print(
    "训练截止:",
    train.order_purchase_timestamp.max(),
    "测试开始:",
    test.order_purchase_timestamp.min(),
)

## 6. 预处理Pipeline与概率基线

类别编码、数值缩放和模型封装为统一流程；Dummy概率作为最低基线。

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.metrics import average_precision_score

preprocess = ColumnTransformer(
    [
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
        ("num", StandardScaler(), num),
    ]
)
dummy = DummyClassifier(strategy="prior").fit(train[features], train.late)
dummy_prob = dummy.predict_proba(val[features])[:, 1]
print(
    "验证集正类率:",
    round(val.late.mean(), 3),
    "Dummy PR-AUC:",
    round(average_precision_score(val.late, dummy_prob), 3),
)

## 7. 比较逻辑回归与随机森林

在同一验证集上比较两个常见分类器，测试集仍保持未查看。

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

models = {
    "逻辑回归": Pipeline(
        [
            ("prep", preprocess),
            (
                "model",
                LogisticRegression(max_iter=700, class_weight="balanced"),
            ),
        ]
    ),
    "随机森林": Pipeline(
        [
            ("prep", preprocess),
            (
                "model",
                RandomForestClassifier(
                    n_estimators=160,
                    min_samples_leaf=8,
                    class_weight="balanced",
                    n_jobs=-1,
                    random_state=106,
                ),
            ),
        ]
    ),
}
rows = []
for name, model in models.items():
    model.fit(train[features], train.late)
    rows.append(
        [
            name,
            average_precision_score(
                val.late, model.predict_proba(val[features])[:, 1]
            ),
        ]
    )
validation = pd.DataFrame(
    rows, columns=["model", "validation_PR_AUC"]
).sort_values("validation_PR_AUC", ascending=False)
display(validation.round(3))
best_name = validation.iloc[0].model
dev = model_df.iloc[:val_end]
best_model = models[best_name].fit(dev[features], dev.late)
probability = best_model.predict_proba(test[features])[:, 1]

## 8. 测试集概率指标与阈值比较

PR-AUC是主指标，并比较不同Top-K比例下的精确率、召回率和Lift。

In [ ]:
from sklearn.metrics import roc_auc_score, log_loss, confusion_matrix

metrics = pd.Series(
    {
        "ROC_AUC": roc_auc_score(test.late, probability),
        "PR_AUC": average_precision_score(test.late, probability),
        "LogLoss": log_loss(test.late, probability),
    }
)
ranked = pd.DataFrame(
    {"actual": test.late.to_numpy(), "probability": probability}
).sort_values("probability", ascending=False)
threshold_rows = []
for share in [0.05, 0.10, 0.20]:
    n = max(1, int(len(ranked) * share))
    top = ranked.head(n)
    threshold_rows.append(
        [
            f"{share:.0%}",
            top.probability.min(),
            top.actual.mean(),
            top.actual.sum() / ranked.actual.sum(),
            top.actual.mean() / ranked.actual.mean(),
        ]
    )
threshold_table = pd.DataFrame(
    threshold_rows, columns=["Top比例", "概率阈值", "Precision", "Recall", "Lift"]
)
print(metrics.round(3).to_string())
display(threshold_table.round(3))
threshold = threshold_table.loc[
    threshold_table["Top比例"] == "10%", "概率阈值"
].iloc[0]
prediction = probability >= threshold
print("Top10%混淆矩阵:", confusion_matrix(test.late, prediction).tolist())

## 9. 错误类型与地区切片

区分漏判与误报，并检查模型在主要客户州的错误率是否一致。

In [ ]:
error_df = test[
    ["customer_state", "promise_days", "item_count", "late"]
].copy()
error_df["probability"] = probability
error_df["prediction"] = prediction
error_df["error_type"] = np.select(
    [
        (error_df.late == 1) & (~error_df.prediction),
        (error_df.late == 0) & error_df.prediction,
    ],
    ["漏判延期", "误报延期"],
    default="判断正确",
)
state_report = (
    error_df.groupby("customer_state")
    .agg(
        orders=("late", "size"),
        late_rate=("late", "mean"),
        mean_score=("probability", "mean"),
        error_rate=("error_type", lambda x: (x != "判断正确").mean()),
    )
    .query("orders>=200")
    .sort_values("error_rate", ascending=False)
)
print(error_df.error_type.value_counts())
display(state_report.head(12).round(3))

## 10. 特征解释与模型局限

用测试子样本计算置换重要性，并讨论数据缺失和预测边界。

In [ ]:
from sklearn.inspection import permutation_importance

sample_n = min(4000, len(test))
sample_idx = np.linspace(0, len(test) - 1, sample_n, dtype=int)
permutation = permutation_importance(
    best_model,
    test.iloc[sample_idx][features],
    test.iloc[sample_idx].late,
    n_repeats=3,
    scoring="average_precision",
    random_state=106,
    n_jobs=-1,
)
importance = pd.Series(
    permutation.importances_mean, index=features
).sort_values(ascending=False)
print("最佳模型:", best_name)
print("置换重要性:\n", importance.round(4))
print("局限: 数据缺少距离、仓库节点、承运商和实时轨迹；地区特征的重要性不能解释为地区导致延期。")

## 项目验收清单

- 完成四表粒度与连接审计
- 订单主键唯一且记录清洗口径
- 排除实际发货、签收和最终状态字段
- 比较Dummy和两个候选模型
- 完成阈值、错误切片和置换重要性分析

建议重新启动内核后从第一个代码单元格运行，确认项目不依赖隐藏状态。

## 本章案例流程

1. 明确订单粒度、预测时点与延期标签
2. 审计四张原始数据表
3. 聚合明细并连接订单级样本
4. 清洗日期并构造延期标签
5. 探索类别不平衡和场景差异
6. 审计泄漏并按时间划分三组数据
7. 建立预处理Pipeline和概率基线
8. 比较逻辑回归与随机森林
9. 评价PR-AUC、阈值、Lift与错误切片
10. 解释特征重要性并总结局限

## 本章案例输出

- 一份从数据审计到模型评价可完整运行的 Notebook
- 数据清洗前后样本变化和关键质量检查结果
- 基线与候选模型的指标对比表
- 错误切片、特征解释和有边界的业务结论

## 阶段检查点

- [ ] 数据与目标定义完成：样本粒度、预测时点和指标已写清楚
- [ ] 基线完成：知道复杂模型相对什么标准比较
- [ ] 模型评价完成：测试集只使用一次，并检查误差切片
- [ ] 交付完成：结论与证据对应，不把相关性写成因果

## 最低完成标准

- 每个代码阶段都有可见输出，不能依赖未展示的隐藏状态。
- 所有关键清洗、筛选和评价口径都写在 Markdown 或注释中。
- 最终结论至少引用一个数值或图表证据，并说明适用范围。

## 提升任务

完成基础验收后，可以增加一个对照方案、一个分组切片或一个参数敏感性实验，比较结果是否稳定。

## 结论与表达

- 多表建模必须先统一到订单粒度
- 标签可使用事后结果，但特征必须在预测时可获得
- 时间顺序划分比随机划分更接近未来预测
- 类别不平衡任务应联合观察PR-AUC、Recall和Lift

## 本章小结

使用 Olist 巴西电商公开数据，以物流延期分类为主线，学习多表建模、预测时点、数据泄漏、类别不平衡和业务阈值。

### 你已经完成

- 理解订单、明细、客户和卖家表的粒度
- 构造订单级延期标签并排除事后字段
- 使用时间顺序划分模拟未来预测
- 比较概率基线、逻辑回归和随机森林
- 使用PR-AUC、Top-K、错误切片与特征重要性评价模型

### 建模流程速查

| 阶段 | 学习内容 |
| --- | --- |
| 步骤 1 | 明确订单粒度、预测时点与延期标签 |
| 步骤 2 | 审计四张原始数据表 |
| 步骤 3 | 聚合明细并连接订单级样本 |
| 步骤 4 | 清洗日期并构造延期标签 |
| 步骤 5 | 探索类别不平衡和场景差异 |
| 步骤 6 | 审计泄漏并按时间划分三组数据 |
| 步骤 7 | 建立预处理Pipeline和概率基线 |
| 步骤 8 | 比较逻辑回归与随机森林 |
| 步骤 9 | 评价PR-AUC、阈值、Lift与错误切片 |
| 步骤 10 | 解释特征重要性并总结局限 |

### 质量与结论提醒

- 订单主键及明细一对多关系
- 聚合连接后订单唯一
- 日期缺失与承诺天数异常
- 多表建模必须先统一到订单粒度
- 标签可使用事后结果，但特征必须在预测时可获得
- 时间顺序划分比随机划分更接近未来预测
- 类别不平衡任务应联合观察PR-AUC、Recall和Lift

### 学习检查

- [ ] 完成四表粒度与连接审计
- [ ] 订单主键唯一且记录清洗口径
- [ ] 排除实际发货、签收和最终状态字段
- [ ] 比较Dummy和两个候选模型
- [ ] 完成阈值、错误切片和置换重要性分析

### 后续迭代建议

完成验收后，记录一个最值得继续验证的假设：可以是更多数据、不同时间窗口、另一种模型，或一个更细的分组分析。